# Working With Geometries

<style>
.cell_output .animation {
  display: block;
  width: fit-content;
  max-width: 100%;
  margin-inline: auto;
}
.cell_output .animation img {
  display: block;
  max-width: 100%;
  height: auto;
  margin: 0 auto;
}
.cell_output .animation .anim-buttons button {
  display: none;
}
.cell_output .animation .anim-buttons button[title="Previous frame"],
.cell_output .animation .anim-buttons button[title="Next frame"] {
  display: inline-block;
  width: 7rem;
  height: auto;
  padding: 0.4rem 0.75rem;
  margin: 0.25rem;
  background: #087fc7;
  border: 0;
  border-radius: 0.25rem;
  color: white;
  cursor: pointer;
}
.cell_output .animation button[title="Previous frame"] i,
.cell_output .animation button[title="Next frame"] i,
.cell_output .animation .anim-state {
  display: none;
}
.cell_output .animation button[title="Previous frame"]::before {
  content: "◀ Back";
}
.cell_output .animation button[title="Next frame"]::before {
  content: "Forward ▶";
}
</style>

In the course of working with legislative redistricting data, it is inevitable that
we will have to work with files that contain geometries. Most often, these geometries
come in the form of shapefiles, which, while nice in theory, can be a bit of a pain to
work with in practice. For now, we will focus on the basics of working with geometries,
but the interested reader is encouraged to explore our partnered library,
[maup](https://github.com/mggg/maup#readme), which is specifically designed to
help fix tricky geometry problems. Specifically, in the event that you are working with
a shapefile and run in to an error of the flavour:

```console
UserWarning: Found overlaps among the given polygons.
Indices of overlaps: {(887, 892), (893, 915), (892, 914), (887, 893)}
```

or

```console
UserWarning: Found islands (degree-0 nodes). Indices of islands: {2552, 3107}
"Found islands (degree-0 nodes). Indices of islands: {}".format(islands)
```

then you should consider consulting the
[maup documentation](https://github.com/mggg/maup/wiki/)
to see if it can help you out.

## Loading and Running a Plan

<div align="center">
  <a href="https://github.com/mggg/GerryChain/tree/main/docs/_static/MN.zip" class="download-badge" download>Download MN File</a>
</div>
<br style="line-height: 5px;">

For this example, we will make use of a Minnesota GeoJSON file that contains
the geometries of the state's precincts (you will need to unzip the above
folder to get to the file -- it's a bit large). We will follow a similar workflow to
what we already covered in the [ReCom section](./recom.ipynb), but with an eye
towards some of the conveniences afforded by `GeographicPartition` objects. As always,
we'll start with the imports:

In [ ]:
import matplotlib.pyplot as plt
from gerrychain import (Partition, Graph, MarkovChain,
                        updaters, constraints, accept,
                        GeographicPartition)
from gerrychain.proposals import build_recom_proposal_fn
from gerrychain.tree import bipartition_tree
from gerrychain.constraints import contiguous
import pandas

And now we load the graph from the GeoJSON file

In [ ]:
import zipfile

with zipfile.ZipFile("MN.zip") as z:
    z.extractall()

graph = Graph.from_file("MN_precincts.geojson")

as well as define our updaters and initial partition

In [ ]:
my_updaters = {
    "population": updaters.Tally("TOTPOP", alias="population"),
    "cut_edges": updaters.cut_edges,
    "perimeter": updaters.perimeter,
    "area": updaters.Tally("area", alias="area"),
}

initial_partition = GeographicPartition(
    graph,
    assignment="CONGDIST",
    updaters=my_updaters
)

The observant reader will notice that we have added two new updaters, `perimeter`,
and `area`, [^1] and we are now using the `GeographicPartition` class instead of the
`Partition` class. The `GeographicPartition` class is a subclass of the
`Partition` class that allows us the capability of working with geometries throughout
our Markov chain, and the `perimeter` and `area` updaters are examples of such a
geometric updater that was previously unavailable to us. These updaters are necessary for
monitoring things like geometric compactness and area via metrics such as the Polsby-Popper
test. [^2]

And now it is time for one of the first conveniences of the `GeographicPartition` class:
we can plot our map and see the initial partition!

In [ ]:
initial_partition.plot()

Of course, this isn't very pretty, so let's pass it some additional arguments to
things a bit nicer:

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.set_yticks([])
ax.set_xticks([])
ax.set_title("Initial Partition in MN")
initial_partition.plot(ax=ax, cmap='tab20c')

Under the hood, the `plot` method is using the `geodataframe.plot` method from
[geopandas](https://geopandas.org/) to plot the geometries, and all of this is
built on top of `matplotlib`, so most of the standard methods for modifying a
`matplotlib` plot will work here as well.

Now that we have our initial partition, we can run a Markov chain on it just as we
have previously:

In [ ]:
ideal_population = sum(initial_partition["population"].values()) / len(initial_partition)


proposal_fn = build_recom_proposal_fn(
    pop_col="TOTPOP",
    pop_target=ideal_population,
    epsilon=0.01,
)

recom_chain = MarkovChain(
    proposal_fn=proposal_fn,
    constraints=[contiguous],
    acceptance_fn=accept.always_accept,
    initial_partition=initial_partition,
    total_steps=20,
    rng=42,
)

The next cell builds an interactive viewer for watching the chain. Its Back and Forward
buttons run entirely in the browser, so they also work in the rendered documentation.

In [ ]:
from io import BytesIO

from matplotlib.animation import ArtistAnimation
from PIL import Image
from IPython.display import HTML

frames = []
district_data = []

for i, partition in enumerate(recom_chain):
    for district_name in partition["perimeter"]:
        district_data.append(
            (
                i,
                district_name,
                partition["population"][district_name],
                partition["perimeter"][district_name],
                partition["area"][district_name],
            )
        )

    with BytesIO() as buffer:
        fig, ax = plt.subplots(figsize=(10, 10))
        partition.plot(ax=ax, cmap="tab20")
        ax.set_xticks([])
        ax.set_yticks([])
        fig.savefig(buffer, format="png", bbox_inches="tight", pad_inches=0)
        frames.append(Image.open(buffer).copy())
        plt.close(fig)

df = pandas.DataFrame(
    district_data,
    columns=["step", "district_name", "population", "perimeter", "area"],
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis("off")
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
artists = [[ax.imshow(frame, animated=True)] for frame in frames]
animation = ArtistAnimation(fig, artists, interval=500)
plt.close(fig)
HTML(animation.to_jshtml())

The dataframe collected the population, perimeter, and area for every district at each step.

In [ ]:
df.head(5)

[^1]: The `area` attribute is added when the graph is built from the GeoDataFrame. The
    `perimeter` updater computes district perimeters from the graph's geometries.
[^2]: The Polsby-Popper test is part of `gerrychain.metrics`.